
# Сверка задолженности `df_port` и `OD` в `df_out`

Ноутбук выполняет две проверки для **первой отчетной даты**:

1. Сверка по УНП:
   - в `df_port` суммируется задолженность по УНП;
   - сумма делится на `1000`;
   - в `df_out` суммируется `OD` первой отчетной даты по УНП;
   - выводятся только отклонения;
   - отдельно отмечаются УНП, которые есть только в одном из DataFrame.

2. Общая сверка:
   - общая задолженность `df_port / 1000`;
   - общая `OD` первой отчетной даты в `df_out`;
   - общее отклонение.

Перед запуском должны существовать DataFrame `df_port` и `df_out`.


In [ ]:
import re
import pandas as pd

## Настройки

In [ ]:

PORT_UNN_COL = "УНП"
PORT_DEBT_COL = "Задолженность"
OUT_UNN_COL = "УНП"


## Сверка

In [ ]:

# Нормализуем УНП
df_port[PORT_UNN_COL] = (
    df_port[PORT_UNN_COL]
    .astype("string")
    .str.strip()
)

df_out[OUT_UNN_COL] = (
    df_out[OUT_UNN_COL]
    .astype("string")
    .str.strip()
)

# Находим первую отчетную дату
report_dates = []

for col in df_out.columns:
    match = re.match(
        r"^задолженность_(\d{2}\.\d{2}\.\d{4})$",
        str(col)
    )

    if match:
        report_dates.append(match.group(1))

if not report_dates:
    raise ValueError(
        "В df_out не найдены столбцы вида "
        "'задолженность_01.01.2026'"
    )

first_date = min(
    report_dates,
    key=lambda x: pd.to_datetime(
        x,
        format="%d.%m.%Y"
    )
)

OUT_OD_COL = f"OD_{first_date}"

if OUT_OD_COL not in df_out.columns:
    raise ValueError(
        f"В df_out отсутствует столбец {OUT_OD_COL}"
    )

print("Первая отчетная дата:", first_date)

# Числовые поля
df_port[PORT_DEBT_COL] = (
    pd.to_numeric(
        df_port[PORT_DEBT_COL],
        errors="coerce"
    )
    .fillna(0)
)

df_out[OUT_OD_COL] = (
    pd.to_numeric(
        df_out[OUT_OD_COL],
        errors="coerce"
    )
    .fillna(0)
)

# df_port: задолженность по УНП / 1000
port_sum = (
    df_port
    .groupby(
        PORT_UNN_COL,
        as_index=False
    )[PORT_DEBT_COL]
    .sum()
)

port_sum[PORT_DEBT_COL] = (
    port_sum[PORT_DEBT_COL] / 1000
)

port_sum = port_sum.rename(
    columns={
        PORT_DEBT_COL: "задолженность_df_port"
    }
)

# df_out: OD по УНП
out_sum = (
    df_out
    .groupby(
        OUT_UNN_COL,
        as_index=False
    )[OUT_OD_COL]
    .sum()
    .rename(
        columns={
            OUT_OD_COL: "OD_df_out"
        }
    )
)

# Объединяем outer, чтобы поймать УНП только в одной из таблиц
debt_check = out_sum.merge(
    port_sum,
    on=OUT_UNN_COL,
    how="outer",
    indicator=True
)

debt_check[
    [
        "OD_df_out",
        "задолженность_df_port"
    ]
] = (
    debt_check[
        [
            "OD_df_out",
            "задолженность_df_port"
        ]
    ]
    .fillna(0)
)

debt_check["отклонение"] = (
    debt_check["OD_df_out"]
    - debt_check["задолженность_df_port"]
)

def get_deviation_status(row):
    if row["_merge"] == "left_only":
        return "УНП есть только в df_out"

    if row["_merge"] == "right_only":
        return "УНП есть только в df_port"

    if abs(row["отклонение"]) > 0.01:
        return "Отклонение задолженности"

    return "Совпадает"

debt_check["статус"] = debt_check.apply(
    get_deviation_status,
    axis=1
)

# Только отклонения
debt_deviations = (
    debt_check.loc[
        debt_check["отклонение"].abs() > 0.01
    ]
    .copy()
)

debt_deviations.drop(
    columns="_merge",
    inplace=True
)

debt_deviations["модуль_отклонения"] = (
    debt_deviations["отклонение"].abs()
)

debt_deviations = (
    debt_deviations
    .sort_values(
        "модуль_отклонения",
        ascending=False
    )
    .drop(
        columns="модуль_отклонения"
    )
    .reset_index(drop=True)
)

print("\\nОтклонения по УНП:")
display(debt_deviations)

# Общая сверка
total_port_debt = (
    df_port[PORT_DEBT_COL].sum() / 1000
)

total_out_od = (
    df_out[OUT_OD_COL].sum()
)

total_deviation = (
    total_out_od - total_port_debt
)

total_debt_check = pd.DataFrame({
    "дата": [first_date],
    "задолженность_df_port": [total_port_debt],
    "OD_df_out": [total_out_od],
    "отклонение": [total_deviation]
})

print("\\nОбщий контроль задолженности:")
display(total_debt_check)

print(f"Первая отчетная дата: {first_date}")
print(
    f"Общая задолженность df_port / 1000: "
    f"{total_port_debt:,.2f}"
)
print(
    f"Общая OD df_out: "
    f"{total_out_od:,.2f}"
)
print(
    f"Общее отклонение: "
    f"{total_deviation:,.2f}"
)
print(
    f"Количество УНП с отклонениями: "
    f"{len(debt_deviations)}"
)



## Результаты

После выполнения основной ячейки доступны:

- `debt_deviations` — отклонения по УНП;
- `total_debt_check` — общий контроль задолженности;
- `first_date` — автоматически определенная первая отчетная дата.


In [ ]:
debt_deviations.head()

In [ ]:
total_debt_check